In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    trim,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_settlement_exposure_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        #schema = (
        #     "SettlementExposureId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp, BatchId string, SettlementExposureKey string, InitialSettlementKey int,"
        #     "PreviouSettlementKey int, SettlementKey int, ExposureKey string, AgreementKey string, PartyKey string, CDI string,"
        #     "CaseCode int, DebtAmount decimal(19,4), PrincipalProposedAmount decimal(19,2), ProposedAmount decimal(19,4),"
        #     "RemainAmount decimal(19,4), SettlementAmount decimal(19,4), ExpectedSettlementAmount decimal(19,4),"
        #     "DiscountAmount decimal(19,4), DiscountPercentage decimal(19,4), BalloonAmount decimal(19,2),"
        #     "BucketCode int, Bucket int, SettlementTypeCode string, SettlementType string, SettlementStatusCode string,"
        #     "SettlementStatus string, CreatedDate timestamp, SubmissionDate timestamp, ApprovalDate date,"
        #     "ImplementationDate timestamp, ExpiredDate timestamp, CancelledDate timestamp, RejectedDate date,"
        #     "OverdueDate date, FirstInstalmentDate date, NextInstalmentDate date, PastInstalmentAmount decimal(19,4),"
        #     "PastInstalmentNumber int, NextInstalmentAmount decimal(19,4), NumOfaccounts int, NumOfinstalments int,"
        #     "TotalPaymentAmount decimal(19,4), IsActiveFlag int, Comments string, KeptPercentage decimal(19,4),"
        #     "IsMigratedFlag boolean, MigrationComments string, PoolName string, AddedManuallyFlag boolean,"
        #     "isPrimarySettlementFlag boolean, ParticipationPercentage decimal(19,4), MaturityPercentage decimal(19,4),"
        #     "MutiplyforLCY decimal(19,4), OtherExpense1 decimal(19,4), OtherExpense2 decimal(19,4), OtherExpense3 decimal(19,4),"
        #     "LastUpdatedDate timestamp, CancellationReasonCode int, CancellationReason string, DiasRFCode string,"
        #     "CtrlMovedFlag boolean,ContractNumber string,CaseFrozenInterestAmtLcy decimal(19,4),CaseProvisionAmtLcy decimal(19,4),"
        #     "MvhoCaseFrozenInterestAmt decimal(19,4),MvhoCaseWriteOffAmt  decimal(19,4), MvhoCaseProvisionAmt  decimal(19,4),"
        #     "SettlementLastInstallmentDate  date, CaseWriteOffAmtLcy  decimal(19,4), StartingOutstandingBalance decimal(19,4), StartingDebtAmount decimal(19,4),"
        #     "StartingFrozen decimal(19,4), StartingWriteOffAmount decimal(19,4), StartingCurrentBucket  int, StartingCaseDenounceStatus string,StartingHostAccountStatus string,
        #     "SepsIsDownPayment boolean,SepsInstallmentAmt  decimal(19,4),AgencyCode string,BusinessDate date"
        # )
#
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.SettlementExposure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "portfolio_id_qca",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ============================== dim_bucket ====================
        dim_bucket_extra_filters = None
        if dim_bucket_extra_filters is not None:
            dim_bucket_extra_filters.append(date_filter)
            dim_bucket_final_filters = " AND ".join(dim_bucket_extra_filters)
        else:
            dim_bucket_final_filters = date_filter
        dim_bucket_selected_fields = [
            "bck_order",
            "bck_id",
        ]
        dim_bucket = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_dim_bucket",
            dim_bucket_final_filters,
            dim_bucket_selected_fields,
        )

        # ============================== dim_cases_static ====================
        dim_cases_static_extra_filters = None
        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "case_account_for_search",
            "case_code",
            "CSTM_TEXT_AD_20",
            "CASE_CONTRACT_ACCOUNT_KEY",
            "CASE_LEGAL_CONTRACT",
            "businessdate",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )
        # ============================== dim_settlement_status ====================
        dim_settlement_status_extra_filters = None
        if dim_settlement_status_extra_filters is not None:
            dim_settlement_status_extra_filters.append(date_filter)
            dim_settlement_status_final_filters = " AND ".join(
                dim_settlement_status_extra_filters
            )
        else:
            dim_settlement_status_final_filters = date_filter
        dim_settlement_status_selected_fields = ["sest_desc", "sest_id"]
        dim_settlement_status = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_dim_settlement_status",
            dim_settlement_status_final_filters,
            dim_settlement_status_selected_fields,
        )

        # ============================== dim_settlement_type ====================
        dim_settlement_type_extra_filters = None
        if dim_settlement_type_extra_filters is not None:
            dim_settlement_type_extra_filters.append(date_filter)
            dim_settlement_type_final_filters = " AND ".join(
                dim_settlement_type_extra_filters
            )
        else:
            dim_settlement_type_final_filters = date_filter
        dim_settlement_type_selected_fields = ["sety_desc", "sety_id"]
        dim_settlement_type = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_dim_settlement_type",
            dim_settlement_type_final_filters,
            dim_settlement_type_selected_fields,
        )

        # ============================== fs_arrangements ====================
        fs_arrangements_extra_filters = None
        if fs_arrangements_extra_filters is not None:
            fs_arrangements_extra_filters.append(date_filter)
            fs_arrangements_final_filters = " AND ".join(fs_arrangements_extra_filters)
        else:
            fs_arrangements_final_filters = date_filter
        fs_arrangements_selected_fields = [
            "sete_code",
            "sety_id",
            "sest_id",
            "sete_bucket",
            "cust_code",
            "sete_setlement_amt",
            "sete_total_proposed_amt",
            "sete_discount_amt",
            "sete_discount_percentage",
            "sete_activation_date",
            "sete_status_last_change_date",
            "sete_last_installment_date",
            "sete_past_installments_amt",
            "sete_past_installments_number",
            "sete_nr_installments",
            "sete_already_paid_amt",
            "sete_comments",
            "sete_kept_percentage",
            "sete_maturity_percentage",
            "sete_multiply_for_lcy",
            "sete_other_expenses_1",
            "sete_other_expenses_2",
            "sete_other_expenses_3",
        ]
        fs_arrangements = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_fs_arrangements",
            fs_arrangements_final_filters,
            fs_arrangements_selected_fields,
        )

        # ============================== fs_arrangements_cases ====================
        fs_arrangements_cases_extra_filters = None
        if fs_arrangements_cases_extra_filters is not None:
            fs_arrangements_cases_extra_filters.append(date_filter)
            fs_arrangements_cases_final_filters = " AND ".join(
                fs_arrangements_cases_extra_filters
            )
        else:
            fs_arrangements_cases_final_filters = date_filter
        fs_arrangements_cases_selected_fields = [
            "indx_active",
            "setc_ratio",
            "sete_code",
            "case_code",
        ]
        fs_arrangements_cases = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_fs_arrangements_cases",
            fs_arrangements_cases_final_filters,
            fs_arrangements_cases_selected_fields,
        )

        # ============================== fs_cases_snap ====================
        fs_cases_snap_extra_filters = None
        if fs_cases_snap_extra_filters is not None:
            fs_cases_snap_extra_filters.append(date_filter)
            fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        else:
            fs_cases_snap_final_filters = date_filter
        fs_cases_snap_selected_fields = [
            "case_debt_amount",
            "case_code",
            "SPET_CODE","CASE_SOURCE",
        ]
        fs_cases_snap = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_fs_cases_snap",
            fs_cases_snap_final_filters,
            fs_cases_snap_selected_fields,
        )

         # ============================== at_cases ====================
        at_cases_extra_filters = None
        if at_cases_extra_filters is not None:
            at_cases_extra_filters.append(date_filter)
            at_cases_final_filters = " AND ".join(at_cases_extra_filters)
        else:
            at_cases_final_filters = date_filter
        at_cases_selected_fields = [
            "CASE_WRITE_OFF_AMT_LCY"
            ,"CASE_PROVISION_AMT_LCY" ,"case_code","CASE_FROZEN_INTEREST_AMT_LCY",
        ]
        at_cases = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_at_cases",
            at_cases_final_filters,
            at_cases_selected_fields,
        )

         # ============================== qca_AT_MOVED_CASES_HOST ====================
        AT_MOVED_CASES_HOST_extra_filters = None
        if AT_MOVED_CASES_HOST_extra_filters is not None:
            AT_MOVED_CASES_HOST_extra_filters.append(date_filter)
            AT_MOVED_CASES_HOST_final_filters = " AND ".join(AT_MOVED_CASES_HOST_extra_filters)
        else:
            AT_MOVED_CASES_HOST_final_filters = date_filter
        AT_MOVED_CASES_HOST_selected_fields = [
            "MVHO_CASE_FROZEN_INTEREST_AMT"
           ,"MVHO_CASE_WRITE_OFF_AMT" 
           ,"MVHO_CASE_PROVISION_AMT","case_code",
        ]
        AT_MOVED_CASES_HOST = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_AT_MOVED_CASES_HOST",
            AT_MOVED_CASES_HOST_final_filters,
            AT_MOVED_CASES_HOST_selected_fields,
        )

         # ============================== qca_AT_SETTLEMENTS_EXT ====================
        AT_SETTLEMENTS_EXT_extra_filters = None
        if AT_SETTLEMENTS_EXT_extra_filters is not None:
            AT_SETTLEMENTS_EXT_extra_filters.append(date_filter)
            AT_SETTLEMENTS_EXT_final_filters = " AND ".join(AT_SETTLEMENTS_EXT_extra_filters)
        else:
            AT_SETTLEMENTS_EXT_final_filters = date_filter
        AT_SETTLEMENTS_EXT_selected_fields = [
         "SETE_CREATED"
         ,"sete_code",
        ]
        AT_SETTLEMENTS_EXT = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_AT_SETTLEMENTS_EXT",
            AT_SETTLEMENTS_EXT_final_filters,
            AT_SETTLEMENTS_EXT_selected_fields,
        )

         # ============================== qca_AT_SETTLEMENTS_EXT_CASES ====================
        AT_SETTLEMENTS_EXT_CASES_extra_filters = None
        if AT_SETTLEMENTS_EXT_CASES_extra_filters is not None:
            AT_SETTLEMENTS_EXT_CASES_extra_filters.append(date_filter)
            AT_SETTLEMENTS_EXT_CASES_final_filters = " AND ".join(AT_SETTLEMENTS_EXT_CASES_extra_filters)
        else:
            AT_SETTLEMENTS_EXT_CASES_final_filters = date_filter
        AT_SETTLEMENTS_EXT_CASES_selected_fields = [
          
                "SETC_ACT_CASE_BUCKET_CODE"
                ,"SETC_ACT_CASE_OUTSTANDING_AMT"
                ,"SETC_ACT_CASE_DEBT_AMT"
                ,"SETC_ACT_CASE_FROZEN_INTEREST_AMT"
                ,"SETC_ACT_CASE_WRITE_OFF_AMT" 
                ,"SETC_ACT_CASE_GENERIC_STATUS_04_INTCODE" 
                ,"SETC_ACT_CASE_GENERIC_STATUS_06_INTCODE","CASE_CODE","SETE_CODE",
        ]
        AT_SETTLEMENTS_EXT_CASES = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_AT_SETTLEMENTS_EXT_CASES",
           AT_SETTLEMENTS_EXT_CASES_final_filters,
            AT_SETTLEMENTS_EXT_CASES_selected_fields,
        )

        # ============================== qca_AT_CASE_CORE ====================
        AT_CASE_CORE_extra_filters = None
        if AT_CASE_CORE_extra_filters is not None:
            AT_CASE_CORE_extra_filters.append(date_filter)
            AT_CASE_CORE_final_filters = " AND ".join(AT_CASE_CORE_extra_filters)
        else:
           AT_CASE_CORE_final_filters = date_filter
        AT_CASE_CORE_selected_fields = [
                "EXTC_CODE",
          "CASE_CODE",
        ]
        AT_CASE_CORE = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_AT_CASE_CORE",
           AT_CASE_CORE_final_filters,
           AT_CASE_CORE_selected_fields,
        )


        # ============================== EOD_CASES_CONTROL ====================
        EOD_CASES_CONTROL_extra_filters = None
        if EOD_CASES_CONTROL_extra_filters is not None:
            EOD_CASES_CONTROL_extra_filters.append(date_filter)
            EOD_CASES_CONTROL_final_filters = " AND ".join(EOD_CASES_CONTROL_extra_filters)
        else:
           EOD_CASES_CONTROL_final_filters = date_filter
        EOD_CASES_CONTROL_selected_fields = [
             "CTRL_MOVED_FLAG" 
            ,"CTRL_CASE_LEGAL_CONTRACT"
            ,"case_code",
        ]
        EOD_CASES_CONTROL = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_EOD_CASES_CONTROL",
           EOD_CASES_CONTROL_final_filters,
           EOD_CASES_CONTROL_selected_fields,
        )

        # ============================== at_external_companies ====================
        at_external_companies_extra_filters = None
        if at_external_companies_extra_filters is not None:
            at_external_companies_extra_filters.append(date_filter)
            at_external_companies_final_filters = " AND ".join(at_external_companies_extra_filters)
        else:
           at_external_companies_final_filters = date_filter
        at_external_companies_selected_fields = [
             "EXTC_CODE"
            ,"EXTC_INTCODE" ,
        ]
        at_external_companies = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_at_external_companies",
           at_external_companies_final_filters,
           at_external_companies_selected_fields,
        )

         # ============================== at_settlements_ext_payments_schedule ====================
        at_settlements_ext_payments_schedule_extra_filters = None
        if at_settlements_ext_payments_schedule_extra_filters is not None:
            at_settlements_ext_payments_schedule_extra_filters.append(date_filter)
            at_settlements_ext_payments_schedule_final_filters = " AND ".join(at_settlements_ext_payments_schedule_extra_filters)
        else:
           at_settlements_ext_payments_schedule_final_filters = date_filter
        at_settlements_ext_payments_schedule_selected_fields = [
          
             "SEPS_IS_DOWN_PAYMENT"
            ,"SEPS_INSTALLMENT_AMT"
            ,"SETE_CODE" ,
        ]
        at_settlements_ext_payments_schedule = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_at_settlements_ext_payments_schedule",
           at_settlements_ext_payments_schedule_final_filters,
           at_settlements_ext_payments_schedule_selected_fields,
        )

        # ============================== dim_customer_static ====================
        dim_customer_static_extra_filters = None
        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(dim_customer_static_extra_filters)
        else:
            dim_customer_static_final_filters = date_filter
        dim_customer_static_selected_fields = [
            "CUST_ID",
            "cust_cif",
        ]
        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )

        # ====================== Rename Columns to avoid conflicts in joined_df ======================
        fs_arrangements = (
            fs_arrangements.withColumnRenamed("sete_code", "fs_arrangements_sete_code")
            .withColumnRenamed("sety_id", "fs_arrangements_sety_id")
            .withColumnRenamed("sest_id", "fs_arrangements_sest_id")
        )

        fs_cases_snap = (fs_cases_snap.withColumnRenamed(
            "case_code", "fs_cases_snap_case_code")
        )

        at_cases=(at_cases.withColumnRenamed( "case_code", "at_cases_case_code"
        ))

        AT_MOVED_CASES_HOST=AT_MOVED_CASES_HOST.withColumnRenamed(
            "case_code", "AT_MOVED_CASES_HOST_case_code"
        )
        at_settlements_ext_payments_schedule=at_settlements_ext_payments_schedule.where(
            at_settlements_ext_payments_schedule.SEPS_IS_DOWN_PAYMENT=='1').withColumnRenamed(
            "sete_code", "at_settlements_ext_payments_schedule_sete_code"
        )

        at_external_companies=at_external_companies.withColumnRenamed(
            "extc_code", "at_external_companies_extc_code"
        )

        AT_SETTLEMENTS_EXT=AT_SETTLEMENTS_EXT.withColumnRenamed(
            "sete_code", "AT_SETTLEMENTS_EXT_sete_code"
        )

        AT_SETTLEMENTS_EXT_CASES=(AT_SETTLEMENTS_EXT_CASES.withColumnRenamed(
            "sete_code", "AT_SETTLEMENTS_EXT_CASES_sete_code").withColumnRenamed("case_code","AT_SETTLEMENTS_EXT_CASES_case_code")
        )

        AT_CASE_CORE=(AT_CASE_CORE.withColumnRenamed(
            "CASE_CODE", "AT_CASE_CORE_CASE_CODE"
        ).withColumnRenamed("EXTC_CODE"   ,"AT_CASE_CORE_EXTC_CODE")
        )

        EOD_CASES_CONTROL=EOD_CASES_CONTROL.withColumnRenamed(
            "case_code", "EOD_CASES_CONTROL_case_code"
        )

        # ================== Join ==================

        # Conditions
        fs_arrangements_fs_arrangements_cases_join_condition = (
            fs_arrangements.fs_arrangements_sete_code == fs_arrangements_cases.sete_code
        )

        fs_arrangements_cases_fs_cases_snap_join_condition = (
            fs_arrangements_cases.case_code == fs_cases_snap.fs_cases_snap_case_code
        )

        fs_cases_snap_dim_cases_static_join_condition = (
            fs_cases_snap.fs_cases_snap_case_code == dim_cases_static.case_code
        )

        fs_arrangements_dim_settlement_type_join_condition = (
            fs_arrangements.fs_arrangements_sety_id == dim_settlement_type.sety_id
        )

        dim_settlement_status_fs_arrangements_join_condition = (
            dim_settlement_status.sest_id == fs_arrangements.fs_arrangements_sest_id
        )

        dim_bucket_fs_arrangements_join_condition = (
            dim_bucket.bck_id == fs_arrangements.sete_bucket
        )

        fs_cases_snap_portfolio_join_condition = (
            fs_cases_snap["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )

        AT_SETTLEMENTS_EXT_CASESat_cases_join_condition = (
             AT_SETTLEMENTS_EXT_CASES.AT_SETTLEMENTS_EXT_CASES_case_code== at_cases.at_cases_case_code
        )

        AT_SETTLEMENTS_EXT_CASES_AT_MOVED_CASES_HOST_join_condition = (
            AT_SETTLEMENTS_EXT_CASES.AT_SETTLEMENTS_EXT_CASES_case_code== AT_MOVED_CASES_HOST.AT_MOVED_CASES_HOST_case_code
        )

        fs_arrangements_AT_SETTLEMENTS_EXT_join_condition = (
            fs_arrangements.fs_arrangements_sete_code== AT_SETTLEMENTS_EXT.AT_SETTLEMENTS_EXT_sete_code
        )

        fs_arrangements_cases_AT_SETTLEMENTS_EXT_CASES_join_condition = (
             (fs_arrangements.fs_arrangements_sete_code== AT_SETTLEMENTS_EXT_CASES.AT_SETTLEMENTS_EXT_CASES_sete_code)
             &
             (fs_cases_snap.fs_cases_snap_case_code== AT_SETTLEMENTS_EXT_CASES.AT_SETTLEMENTS_EXT_CASES_case_code)
        )

        AT_SETTLEMENTS_EXT_CASES_AT_CASE_CORE_join_condition = (
            AT_SETTLEMENTS_EXT_CASES.AT_SETTLEMENTS_EXT_CASES_case_code== AT_CASE_CORE.AT_CASE_CORE_CASE_CODE
        )

        AT_SETTLEMENTS_EXT_CASES_EOD_CASES_CONTROL_join_condition = (
             AT_SETTLEMENTS_EXT_CASES.AT_SETTLEMENTS_EXT_CASES_case_code== EOD_CASES_CONTROL.EOD_CASES_CONTROL_case_code
        )

        AT_SETTLEMENTS_EXT_at_settlements_ext_payments_schedule_join_condition = (
            AT_SETTLEMENTS_EXT.AT_SETTLEMENTS_EXT_sete_code== at_settlements_ext_payments_schedule.at_settlements_ext_payments_schedule_sete_code
        )

        at_cases_at_external_companies_join_condition = (
            AT_CASE_CORE.AT_CASE_CORE_EXTC_CODE== at_external_companies.at_external_companies_extc_code
        )

        fs_arrangements_dim_customer_static_join_condition = (
            fs_arrangements.cust_code == dim_customer_static.CUST_ID
        )

        # Join
        joined_df = (
            fs_arrangements.join(
                fs_arrangements_cases,
                fs_arrangements_fs_arrangements_cases_join_condition,
                "inner",
            )
            .join(
                fs_cases_snap,
                fs_arrangements_cases_fs_cases_snap_join_condition,
                "inner",
            )
            .join(
                dim_cases_static, fs_cases_snap_dim_cases_static_join_condition, "inner"
            )
            .join(
                dim_settlement_type,
                fs_arrangements_dim_settlement_type_join_condition,
                "inner",
            )
            .join(
                dim_settlement_status,
                dim_settlement_status_fs_arrangements_join_condition,
                "left",
            )
            .join(dim_bucket, dim_bucket_fs_arrangements_join_condition, "left")
            .join(
                portfolio,
                fs_cases_snap_portfolio_join_condition,
                "left",
            )
            .join(
                AT_SETTLEMENTS_EXT,
                fs_arrangements_AT_SETTLEMENTS_EXT_join_condition
                , "left"
            )
            .join(
                AT_SETTLEMENTS_EXT_CASES,
                fs_arrangements_cases_AT_SETTLEMENTS_EXT_CASES_join_condition
                , "inner"
                )
            .join(
                at_cases,
                AT_SETTLEMENTS_EXT_CASESat_cases_join_condition
                , "left"
            )
            .join(
                AT_CASE_CORE,
                AT_SETTLEMENTS_EXT_CASES_AT_CASE_CORE_join_condition
                , "left"
            )
            .join(
                EOD_CASES_CONTROL,
                AT_SETTLEMENTS_EXT_CASES_EOD_CASES_CONTROL_join_condition
                , "left")
            .join(
                AT_MOVED_CASES_HOST,
                AT_SETTLEMENTS_EXT_CASES_AT_MOVED_CASES_HOST_join_condition
                , "left"
            )
            .join(
                at_settlements_ext_payments_schedule,
                AT_SETTLEMENTS_EXT_at_settlements_ext_payments_schedule_join_condition
                , "left"
            )
            .join(
                at_external_companies,
                at_cases_at_external_companies_join_condition
                , "left"
           
        )
            .join(
                dim_customer_static,
                fs_arrangements_dim_customer_static_join_condition
                , "left"
        )
        )

        # ====================== HashKey ======================

        joined_df_primary_key_fields = [
            joined_df.fs_arrangements_sete_code,
            joined_df.case_account_for_search,
            lit("qca"),
        ]

        joined_df = (
            joined_df.withColumn(
                "SettlementExposureId",
                sha2(concat_ws(";", *joined_df_primary_key_fields), 256),
            )
            .withColumn(
                "SettlementExposureKey",
                concat_ws(";", "fs_arrangements_sete_code", "case_account_for_search"),
            )
            .withColumn(
                "ImplementationDate",
                when(
                    col("sest_desc") == "Fulfilled",
                    col("sete_status_last_change_date")
                ).otherwise(lit(None).cast("timestamp"))
            ) \
            .withColumn(
                "CancelledDate",
                when(
                    col("sest_desc").startswith("Cancelled"),
                    col("sete_status_last_change_date")
                ).otherwise(lit(None).cast("timestamp"))
            )
            .withColumn(
                "Bucket",
                when(
                    coalesce(dim_bucket.bck_order, fs_arrangements.sete_bucket) == -1, 0
                )
                .when(
                    coalesce(dim_bucket.bck_order, fs_arrangements.sete_bucket) == -100,
                    None,
                )
                .otherwise(coalesce(dim_bucket.bck_order, fs_arrangements.sete_bucket)),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_qca = joined_df.select(
            joined_df.SettlementExposureId.cast("string"),
            lit("qca").cast("string").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            joined_df.SettlementExposureKey.cast("string"),
            lit(None).cast("int").alias("InitialSettlementKey"),
            lit(None).cast("int").alias("PreviouSettlementKey"),
            joined_df.fs_arrangements_sete_code.cast("int").alias("SettlementKey"),
            joined_df.case_account_for_search.cast("string").alias("ExposureKey"),
            when(
                dim_cases_static.CSTM_TEXT_AD_20.isNull() | (trim(dim_cases_static.CSTM_TEXT_AD_20) == ""),
                when(
                    (fs_cases_snap.CASE_SOURCE).like("C%"),
                    dim_cases_static.CASE_CONTRACT_ACCOUNT_KEY,
                ).otherwise(dim_cases_static.CASE_LEGAL_CONTRACT)
            ).otherwise(dim_cases_static.CSTM_TEXT_AD_20).cast("string").alias("AgreementKey"),
            joined_df.cust_code.cast("string").alias("PartyKey"),
            dim_customer_static.cust_cif.cast("string").alias("CDI"),
            fs_arrangements_cases.case_code.cast("int").alias("CaseCode"),
            joined_df.case_debt_amount.cast("decimal(19,4)").alias("DebtAmount"),
            lit(None).cast("decimal(19,2)").alias("PrincipalProposedAmount"),
            joined_df.sete_total_proposed_amt.cast("decimal(19,4)").alias("ProposedAmount"),
            lit(None).cast("decimal(19,4)").alias("RemainAmount"),
            joined_df.sete_setlement_amt.cast("decimal(19,4)").alias(
                "SettlementAmount"
            ),
            joined_df.sete_total_proposed_amt.cast("decimal(19,4)").alias(
                "ExpectedSettlementAmount"
            ),
            joined_df.sete_discount_amt.cast("decimal(19,4)").alias("DiscountAmount"),
            joined_df.sete_discount_percentage.cast("decimal(19,4)").alias(
                "DiscountPercentage"
            ),
            lit(None).cast("decimal(19,2)").alias("BalloonAmount"),
            joined_df.sete_bucket.cast("int").alias("BucketCode"),
            joined_df.Bucket.cast("int"),
            joined_df.fs_arrangements_sety_id.cast("string").alias(
                "SettlementTypeCode"
            ),
            joined_df.sety_desc.cast("string").alias("SettlementType"),
            joined_df.fs_arrangements_sest_id.cast("string").alias(
                "SettlementStatusCode"
            ),
            joined_df.sest_desc.cast("string").alias("SettlementStatus"),
            joined_df.SETE_CREATED.cast("timestamp").alias("CreatedDate"),
            joined_df.sete_activation_date.cast("timestamp").alias("SubmissionDate"),
            lit(None).cast("date").alias("ApprovalDate"),
            joined_df.ImplementationDate.cast("timestamp"),
            joined_df.sete_last_installment_date.cast("timestamp").alias("ExpiredDate"),
            joined_df.CancelledDate.cast("timestamp"),
            lit(None).cast("date").alias("RejectedDate"),
            lit(None).cast("date").alias("OverdueDate"),
            lit(None).cast("date").alias("FirstInstalmentDate"),
            lit(None).cast("date").alias("NextInstalmentDate"),
            joined_df.sete_past_installments_amt.cast("decimal(19,4)").alias(
                "PastInstalmentAmount"
            ),
            joined_df.sete_past_installments_number.cast("int").alias(
                "PastInstalmentNumber"
            ),
            lit(None).cast("decimal(19,4)").alias("NextInstalmentAmount"),
            lit(None).cast("int").alias("NumOfaccounts"),
            joined_df.sete_nr_installments.cast("int").alias("NumOfinstalments"),
            joined_df.sete_already_paid_amt.cast("decimal(19,4)").alias(
                "TotalPaymentAmount"
            ),
            joined_df.indx_active.cast("int").alias("IsActiveFlag"),
            joined_df.sete_comments.cast("string").alias("Comments"),
            joined_df.sete_kept_percentage.cast("decimal(19,4)").alias(
                "KeptPercentage"
            ),
            lit(None).cast("boolean").alias("IsMigratedFlag"),
            lit(None).cast("string").alias("MigrationComments"),
            lit(None).cast("string").alias("PoolName"),
            lit(None).cast("boolean").alias("AddedManuallyFlag"),
            lit(None).cast("boolean").alias("isPrimarySettlementFlag"),
            joined_df.setc_ratio.cast("decimal(19,4)").alias("ParticipationPercentage"),
            joined_df.sete_maturity_percentage.cast("decimal(19,4)").alias(
                "MaturityPercentage"
            ),
            joined_df.sete_multiply_for_lcy.cast("decimal(19,4)").alias(
                "MutiplyforLCY"
            ),
            joined_df.sete_other_expenses_1.cast("decimal(19,4)").alias(
                "OtherExpense1"
            ),
            joined_df.sete_other_expenses_2.cast("decimal(19,4)").alias(
                "OtherExpense2"
            ),
            joined_df.sete_other_expenses_3.cast("decimal(19,4)").alias(
                "OtherExpense3"
            ),
            joined_df.sete_status_last_change_date.cast("timestamp").alias(
                "LastUpdatedDate"
            ),
            lit(None).cast("int").alias("CancellationReasonCode"),
            lit(None).cast("string").alias("CancellationReason"),
            lit(None).cast("string").alias("DiasRFCode")
            ,dim_cases_static.businessdate.cast("date").alias("BusinessDate")
            ,joined_df.MVHO_CASE_FROZEN_INTEREST_AMT.alias("MvhoCaseFrozenInterestAmt")
            ,joined_df.MVHO_CASE_WRITE_OFF_AMT.alias("MvhoCaseWriteOffAmt")
            ,joined_df.MVHO_CASE_PROVISION_AMT.alias("MvhoCaseProvisionAmt")
            ,joined_df.sete_last_installment_date.cast("date").alias("SettlementLastInstallmentDate")
            ,joined_df.CASE_WRITE_OFF_AMT_LCY.alias("CaseWriteOffAmtLcy")
            ,joined_df.SETC_ACT_CASE_OUTSTANDING_AMT.alias("StartingOutstandingBalance")
            ,joined_df.SETC_ACT_CASE_DEBT_AMT.alias("StartingDebtAmount")
            ,joined_df.SETC_ACT_CASE_FROZEN_INTEREST_AMT.alias("StartingFrozen")
            ,joined_df.SETC_ACT_CASE_WRITE_OFF_AMT.alias("StartingWriteOffAmount")
            ,joined_df.SETC_ACT_CASE_BUCKET_CODE.alias("StartingCurrentBucket")
            ,joined_df.SETC_ACT_CASE_GENERIC_STATUS_04_INTCODE.alias("StartingCaseDenounceStatus")
            ,joined_df.SETC_ACT_CASE_GENERIC_STATUS_06_INTCODE.alias("StartingHostAccountStatus")
            ,joined_df.SEPS_IS_DOWN_PAYMENT.alias("SepsIsDownPayment")
            ,joined_df.SEPS_INSTALLMENT_AMT.alias("SepsInstallmentAmt")
            ,joined_df.EXTC_INTCODE.alias("AgencyCode")
            ,joined_df.CTRL_MOVED_FLAG.alias("CtrlMovedFlag")
            ,joined_df.CTRL_CASE_LEGAL_CONTRACT.alias("ContractNumber")
            ,joined_df.CASE_FROZEN_INTEREST_AMT_LCY.alias("CaseFrozenInterestAmtLcy")
            ,joined_df.CASE_PROVISION_AMT_LCY.alias("CaseProvisionAmtLcy")
            ,lit(None).cast("int").alias("DaysPastDue"),
        )

        # ================== Append =================
        result_df_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
             and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_settlement_exposure_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)